# Vaani Paired v2: ARTPARK vs Adalat Smoke Evaluation

This GPU Colab notebook evaluates two pinned Hindi Whisper checkpoints through one inference and multi-reference scoring path:

- `ARTPARK-IISc/whisper-medium-vaani-hindi`
- `adalat-ai/whisper-small-hi-high-lr`

Default `smoke` profile uses 10 speakers across all five frozen v2 channel conditions: 100 total transcriptions. Every prediction is appended directly to Google Drive, so interrupted runs resume. Do not change to `full` until smoke output is reviewed.

In [ ]:
import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
if subprocess.run(['nvidia-smi'], check=False).returncode:
    raise RuntimeError('GPU runtime required. Select Runtime > Change runtime type > T4 GPU.')

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')
REPO_DIR = Path('/content/CallWhisper-8k')
os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/anshulLuhsna/CallWhisper-8k.git', str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)
SRC_DIR = REPO_DIR / 'src'
sys.path.insert(0, str(SRC_DIR))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.46,<5', 'accelerate>=1,<2', 'jiwer>=3,<5',
    'pandas>=2', 'soundfile>=0.12', 'tabulate>=0.9', 'tqdm>=4.66',
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
importlib.invalidate_caches()
_multiref = importlib.import_module('callwhisper.eval.multiref')

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Python:', platform.python_version())
print('Repository commit:', commit)
print('Scorer:', _multiref.__file__)

In [ ]:
from callwhisper.datasets.paired_telephony import CONDITIONS

RUN_PROFILE = 'smoke'  # Keep smoke until its output is reviewed.
PROFILE_SPEAKERS = {'smoke': 10, 'full': 500}
if RUN_PROFILE not in PROFILE_SPEAKERS:
    raise ValueError(f'Unknown profile: {RUN_PROFILE}')
SPEAKER_LIMIT = PROFILE_SPEAKERS[RUN_PROFILE]
NUM_BEAMS = 1
LANGUAGE = 'hi'
TASK = 'transcribe'
SEED = 0
MODELS = (
    {
        'label': 'artpark_medium_vaani_hindi',
        'model_id': 'ARTPARK-IISc/whisper-medium-vaani-hindi',
        'revision': '8e4d906e0eec66f27a31286e1a034702ef6d11bc',
    },
    {
        'label': 'adalat_whisper_small_hi_high_lr',
        'model_id': 'adalat-ai/whisper-small-hi-high-lr',
        'revision': 'e78553113fe7a483dbf82fefb2cbe4ea4b6bf901',
    },
)

V2_INPUT = DRIVE_PROJECT_DIR / 'results/vaani_paired_pilot_v2'
WORK_ROOT = Path('/content/vaani_paired_eval_v2')
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'results' / f'vaani_paired_model_{RUN_PROFILE}_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

semantic_config = {
    'run_profile': RUN_PROFILE,
    'speaker_limit': SPEAKER_LIMIT,
    'conditions': list(CONDITIONS),
    'models': list(MODELS),
    'num_beams': NUM_BEAMS,
    'language': LANGUAGE,
    'task': TASK,
    'seed': SEED,
    'dataset_revision': '1bf019521d12d742178acc32bf2a42f81cf7c8ef',
    'paired_artifact': 'vaani_paired_pilot_v2',
}
config_path = OUTPUT_DIR / 'run_config.json'
if config_path.exists():
    existing = json.loads(config_path.read_text(encoding='utf-8'))
    if existing['semantic_config'] != semantic_config:
        raise RuntimeError(f'Existing output has different config: {config_path}')
config_path.write_text(
    json.dumps({'semantic_config': semantic_config, 'repo_commit': commit}, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(semantic_config, indent=2))
print('Persistent output:', OUTPUT_DIR)

In [ ]:
import tarfile

import pandas as pd
from tqdm.auto import tqdm


def safe_extract(archive_path: Path, destination: Path) -> None:
    root = destination.resolve()
    with tarfile.open(archive_path, 'r:gz') as archive:
        members = archive.getmembers()
        for member in members:
            if member.issym() or member.islnk():
                raise RuntimeError(f'Refusing archive link: {member.name}')
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe archive path: {member.name}')
        for member in tqdm(members, desc=f'Restoring {archive_path.name}'):
            archive.extract(member, destination, filter='data')


for condition in CONDITIONS:
    condition_dir = WORK_ROOT / 'paired_audio' / condition
    existing_files = list(condition_dir.glob('*.wav')) if condition_dir.exists() else []
    if len(existing_files) < 500:
        archive_path = V2_INPUT / 'archives' / f'{condition}.tar.gz'
        if not archive_path.exists():
            raise FileNotFoundError(archive_path)
        safe_extract(archive_path, WORK_ROOT)
    print(condition, 'files=', len(list(condition_dir.glob('*.wav'))))

manifest_path = V2_INPUT / 'vaani_pilot_500_all_conditions.csv'
pilot_path = V2_INPUT / 'vaani_pilot_500.csv'
if not manifest_path.exists() or not pilot_path.exists():
    raise FileNotFoundError('V2 manifests are missing from Drive')
paired_df = pd.read_csv(manifest_path)
pilot_df = pd.read_csv(pilot_path)
sample_keys = pilot_df['sample_key'].astype(str).head(SPEAKER_LIMIT).tolist()
eval_df = paired_df[paired_df['sample_key'].astype(str).isin(sample_keys)].copy()
eval_df['sample_key'] = eval_df['sample_key'].astype(str)
eval_df['condition'] = pd.Categorical(eval_df['condition'], categories=CONDITIONS, ordered=True)
eval_df['sample_order'] = eval_df['sample_key'].map({key: i for i, key in enumerate(sample_keys)})
eval_df = eval_df.sort_values(['sample_order', 'condition']).reset_index(drop=True)
eval_df['resolved_audio_path'] = eval_df['audio_path'].map(lambda path: str(WORK_ROOT / path))

assert len(eval_df) == SPEAKER_LIMIT * len(CONDITIONS)
assert eval_df.groupby('sample_key')['condition'].nunique().eq(len(CONDITIONS)).all()
assert all(Path(path).exists() for path in eval_df['resolved_audio_path'])
assert {'reference_1', 'reference_2', 'reference_3'}.issubset(eval_df.columns)
eval_df.to_csv(OUTPUT_DIR / 'frozen_eval_rows.csv', index=False)
print('Speakers:', eval_df['sample_key'].nunique())
print('Evaluation rows per model:', len(eval_df))
display(eval_df[['sample_key', 'condition', 'gender', 'state']].head(10))

In [ ]:
from callwhisper.eval.multiref import (
    alignment_multireference_score,
    normalize_vaani_text,
)

alternative = alignment_multireference_score(
    ['a b', 'a x b', 'a b'],
    'a x b',
)
unanimous_deletion = alignment_multireference_score(
    ['a x b', 'a x b', 'a x b'],
    'a b',
)
assert alternative.wer == 0.0
assert unanimous_deletion.deletions == 1
assert round(unanimous_deletion.wer, 6) == round(1 / 3, 6)
print('Multi-reference scorer sanity checks passed.')
print('First normalized references:')
first = eval_df.iloc[0]
for column in ('reference_1', 'reference_2', 'reference_3'):
    print(column, '=>', normalize_vaani_text(first[column]))

In [ ]:
import gc
import time

import soundfile as sf
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('CUDA GPU is required')
DTYPE = torch.float16
print('GPU:', torch.cuda.get_device_name(0))


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSONL at {path}:{line_number}') from exc
    return rows


def append_jsonl(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
        handle.flush()


def transcribe_one(model, processor, audio_path: Path) -> tuple[str, float]:
    audio, sample_rate = sf.read(audio_path, dtype='float32', always_2d=False)
    if sample_rate != 16000:
        raise ValueError(f'Expected 16 kHz, got {sample_rate}: {audio_path}')
    if getattr(audio, 'ndim', 1) != 1:
        raise ValueError(f'Expected mono audio: {audio_path}')
    inputs = processor(audio, sampling_rate=sample_rate, return_tensors='pt')
    input_features = inputs.input_features.to(device=DEVICE, dtype=DTYPE)
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        predicted_ids = model.generate(
            input_features=input_features,
            language=LANGUAGE,
            task=TASK,
            num_beams=NUM_BEAMS,
            do_sample=False,
            max_new_tokens=225,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
    return text, elapsed


def evaluate_model(spec: dict) -> Path:
    output_path = OUTPUT_DIR / f"{spec['label']}_predictions.jsonl"
    existing = read_jsonl(output_path)
    complete = {(row['sample_key'], row['condition']) for row in existing}
    expected = {(row.sample_key, str(row.condition)) for row in eval_df.itertuples()}
    if complete == expected:
        print(spec['label'], 'already complete; skipping model load.')
        return output_path
    if not complete.issubset(expected):
        raise RuntimeError(f'Unexpected rows in checkpoint: {output_path}')

    print('Loading', spec['model_id'], 'revision', spec['revision'])
    processor = AutoProcessor.from_pretrained(spec['model_id'], revision=spec['revision'])
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        spec['model_id'],
        revision=spec['revision'],
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    ).to(DEVICE)
    model.eval()
    model.generation_config.forced_decoder_ids = None

    pending = [
        row for row in eval_df.to_dict('records')
        if (row['sample_key'], str(row['condition'])) not in complete
    ]
    for row in tqdm(pending, desc=spec['label']):
        hypothesis, elapsed = transcribe_one(
            model,
            processor,
            Path(row['resolved_audio_path']),
        )
        references = [row['reference_1'], row['reference_2'], row['reference_3']]
        score = alignment_multireference_score(references, hypothesis)
        duration = float(row['duration_s'])
        append_jsonl(output_path, {
            'model_label': spec['label'],
            'model_id': spec['model_id'],
            'model_revision': spec['revision'],
            'sample_key': row['sample_key'],
            'speaker_id': str(row['speaker_id']),
            'gender': row['gender'],
            'state': row['state'],
            'district': row['district'],
            'condition': str(row['condition']),
            'audio_path': row['audio_path'],
            'reference_1': row['reference_1'],
            'reference_2': row['reference_2'],
            'reference_3': row['reference_3'],
            'hypothesis_text': hypothesis,
            'substitutions': score.substitutions,
            'insertions': score.insertions,
            'deletions': score.deletions,
            'errors': score.errors,
            'reference_words': score.reference_words,
            'multiref_wer': score.wer,
            'duration_s': duration,
            'inference_s': elapsed,
            'real_time_factor': elapsed / duration if duration > 0 else None,
            'num_beams': NUM_BEAMS,
            'language': LANGUAGE,
            'repo_commit': commit,
        })

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print('Completed:', output_path, 'rows=', len(read_jsonl(output_path)))
    return output_path

In [ ]:
prediction_paths = [evaluate_model(spec) for spec in MODELS]
print('Prediction checkpoints:')
for path in prediction_paths:
    print('-', path)

In [ ]:
prediction_rows = [row for path in prediction_paths for row in read_jsonl(path)]
predictions_df = pd.DataFrame(prediction_rows)
expected_total = len(MODELS) * SPEAKER_LIMIT * len(CONDITIONS)
assert len(predictions_df) == expected_total
assert not predictions_df.duplicated(['model_label', 'sample_key', 'condition']).any()


def summarize_group(frame: pd.DataFrame, label: str) -> dict:
    substitutions = int(frame['substitutions'].sum())
    insertions = int(frame['insertions'].sum())
    deletions = int(frame['deletions'].sum())
    reference_words = int(frame['reference_words'].sum())
    return {
        'model_label': frame['model_label'].iloc[0],
        'slice': label,
        'speakers': int(frame['sample_key'].nunique()),
        'files': int(len(frame)),
        'substitutions': substitutions,
        'insertions': insertions,
        'deletions': deletions,
        'reference_words': reference_words,
        'multiref_corpus_wer': (substitutions + insertions + deletions) / reference_words,
        'macro_utterance_wer': float(frame['multiref_wer'].mean()),
        'mean_real_time_factor': float(frame['real_time_factor'].mean()),
    }


summary_rows = []
for model_label, model_frame in predictions_df.groupby('model_label', sort=False):
    for condition in CONDITIONS:
        condition_frame = model_frame[model_frame['condition'] == condition]
        summary_rows.append(summarize_group(condition_frame, condition))
    telephone_frame = model_frame[model_frame['condition'] != 'original']
    summary_rows.append(summarize_group(telephone_frame, 'pooled_telephone'))

summary_df = pd.DataFrame(summary_rows)
comparison_df = summary_df.pivot(
    index='slice', columns='model_label', values='multiref_corpus_wer'
).reset_index()
model_labels = [spec['label'] for spec in MODELS]
comparison_df['adalat_minus_artpark_wer'] = (
    comparison_df[model_labels[1]] - comparison_df[model_labels[0]]
)

summary_csv = OUTPUT_DIR / 'summary.csv'
summary_md = OUTPUT_DIR / 'summary.md'
comparison_csv = OUTPUT_DIR / 'artpark_vs_adalat.csv'
comparison_md = OUTPUT_DIR / 'artpark_vs_adalat.md'
summary_df.to_csv(summary_csv, index=False)
summary_md.write_text(summary_df.to_markdown(index=False) + '\n', encoding='utf-8')
comparison_df.to_csv(comparison_csv, index=False)
comparison_md.write_text(comparison_df.to_markdown(index=False) + '\n', encoding='utf-8')

print('Per-condition multi-reference summary:')
display(summary_df)
print('ARTPARK vs Adalat (positive delta means Adalat has higher WER):')
display(comparison_df)
print('Saved under:', OUTPUT_DIR)

## Stop After Smoke

Send back `summary.csv` or the displayed tables. Smoke output checks pipeline correctness and rough behavior only; 10 speakers are not enough for benchmark claims. Do not switch `RUN_PROFILE` to `full` until prediction quality and row completeness are reviewed.